# SIFT vs ORB: Feature Detector Comparison for Image Stitching

This notebook compares two feature extraction algorithms on the same stitching task:

- **SIFT** (Scale-Invariant Feature Transform) — scale + rotation invariant, patented, slower
- **ORB** (Oriented FAST and Rotated BRIEF) — rotation invariant only, open-source, faster

**Why this matters:** The choice of feature detector directly impacts matching quality,
RANSAC convergence speed, and final stitching accuracy. Understanding the trade-off is
essential engineering knowledge.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

from src.features import extract_sift, match_features, draw_matches, draw_keypoints
from src.homography import compute_homography, mean_reprojection_error

rcParams['figure.dpi'] = 120
rcParams['font.size'] = 12
print('✅ Imports OK')

## 1. Load Test Images

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data', 'yard')
img1 = cv2.imread(os.path.join(data_dir, 'im01.jpg'))
img2 = cv2.imread(os.path.join(data_dir, 'im02.jpg'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0].set_title('Image 1', fontsize=14)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[1].set_title('Image 2', fontsize=14)
axes[1].axis('off')
plt.show()

## 2. Extract Features: SIFT vs ORB

In [ ]:
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# --- SIFT ---
t0 = time.time()
sift = cv2.SIFT_create()
kp1_sift, desc1_sift = sift.detectAndCompute(gray1, None)
kp2_sift, desc2_sift = sift.detectAndCompute(gray2, None)
t_sift = time.time() - t0

# --- ORB ---
t0 = time.time()
orb = cv2.ORB_create(nfeatures=2000)
kp1_orb, desc1_orb = orb.detectAndCompute(gray1, None)
kp2_orb, desc2_orb = orb.detectAndCompute(gray2, None)
t_orb = time.time() - t0

print(f"{'':>8} {'Keypoints 1':>12} {'Keypoints 2':>12} {'Time':>10}")
print(f"{'SIFT':>8} {len(kp1_sift):>12} {len(kp2_sift):>12} {t_sift:>9.3f}s")
print(f"{'ORB':>8} {len(kp1_orb):>12} {len(kp2_orb):>12} {t_orb:>9.3f}s")
print(f"\nSpeedup: {t_sift/t_orb:.1f}× (ORB faster)")

## 3. Visualize Keypoint Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# SIFT keypoints
vis_s1 = cv2.drawKeypoints(gray1, kp1_sift[:500], None,
                           color=(0,255,255),
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis_s2 = cv2.drawKeypoints(gray2, kp2_sift[:500], None,
                           color=(0,255,255),
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

axes[0,0].imshow(vis_s1)
axes[0,0].set_title(f'SIFT: {len(kp1_sift)} keypoints (img 1)', fontsize=13)
axes[0,0].axis('off')
axes[0,1].imshow(vis_s2)
axes[0,1].set_title(f'SIFT: {len(kp2_sift)} keypoints (img 2)', fontsize=13)
axes[0,1].axis('off')

# ORB keypoints
vis_o1 = cv2.drawKeypoints(gray1, kp1_orb[:500], None,
                           color=(0,255,0),
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis_o2 = cv2.drawKeypoints(gray2, kp2_orb[:500], None,
                           color=(0,255,0),
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

axes[1,0].imshow(vis_o1)
axes[1,0].set_title(f'ORB: {len(kp1_orb)} keypoints (img 1)', fontsize=13)
axes[1,0].axis('off')
axes[1,1].imshow(vis_o2)
axes[1,1].set_title(f'ORB: {len(kp2_orb)} keypoints (img 2)', fontsize=13)
axes[1,1].axis('off')

plt.suptitle('Keypoint Distribution: SIFT (top) vs ORB (bottom)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature Matching: FLANN (SIFT) vs Brute-Force Hamming (ORB)

In [ ]:
# --- SIFT matching (FLANN + ratio test) ---
sift_match = match_features(desc1_sift, desc2_sift, ratio_thresh=0.75, cross_check=True)

# --- ORB matching (Brute-Force Hamming + ratio test) ---
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
matches_raw_orb = bf.knnMatch(desc1_orb, desc2_orb, k=2)

# Lowe's ratio test for ORB
good_orb = []
for m, n in matches_raw_orb:
    if m.distance < 0.75 * n.distance:
        good_orb.append(m)

print(f"{'':>8} {'Matches':>10} {'After Ratio Test':>18}")
print(f"{'SIFT':>8} {len(sift_match.all_matches):>10} {len(sift_match.good_matches):>18}")
print(f"{'ORB':>8} {len(matches_raw_orb):>10} {len(good_orb):>18}")

## 5. Match Visualization

In [ ]:
# Draw matches side by side
h1, w1 = gray1.shape
h2, w2 = gray2.shape

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# SIFT matches
canvas_s = np.zeros((max(h1,h2), w1+w2, 3), dtype=np.uint8)
canvas_s[:h1, :w1] = cv2.cvtColor(gray1, cv2.COLOR_GRAY2BGR)
canvas_s[:h2, w1:] = cv2.cvtColor(gray2, cv2.COLOR_GRAY2BGR)

show_n = min(40, len(sift_match.good_matches))
indices = np.random.choice(len(sift_match.good_matches), show_n, replace=False)
for i in indices:
    m = sift_match.good_matches[i]
    pt1 = tuple(map(int, kp1_sift[m.queryIdx].pt))
    pt2 = (int(kp2_sift[m.trainIdx].pt[0]) + w1, int(kp2_sift[m.trainIdx].pt[1]))
    color = tuple(np.random.randint(0,255,3).tolist())
    cv2.line(canvas_s, pt1, pt2, color, 1, cv2.LINE_AA)

axes[0].imshow(canvas_s)
axes[0].set_title(f'SIFT Matches: {len(sift_match.good_matches)} pairs', fontsize=14)
axes[0].axis('off')

# ORB matches
canvas_o = np.zeros((max(h1,h2), w1+w2, 3), dtype=np.uint8)
canvas_o[:h1, :w1] = cv2.cvtColor(gray1, cv2.COLOR_GRAY2BGR)
canvas_o[:h2, w1:] = cv2.cvtColor(gray2, cv2.COLOR_GRAY2BGR)

show_n_o = min(40, len(good_orb))
if show_n_o > 0:
    indices_o = np.random.choice(len(good_orb), show_n_o, replace=False)
    for i in indices_o:
        m = good_orb[i]
        pt1 = tuple(map(int, kp1_orb[m.queryIdx].pt))
        pt2 = (int(kp2_orb[m.trainIdx].pt[0]) + w1, int(kp2_orb[m.trainIdx].pt[1]))
        color = tuple(np.random.randint(0,255,3).tolist())
        cv2.line(canvas_o, pt1, pt2, color, 1, cv2.LINE_AA)

axes[1].imshow(canvas_o)
axes[1].set_title(f'ORB Matches: {len(good_orb)} pairs', fontsize=14)
axes[1].axis('off')

plt.suptitle('Feature Matching Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. RANSAC Homography: Which gives better geometry?

In [ ]:
from src.ransac import find_homography_ransac

# --- SIFT RANSAC ---
src_s = np.float32([kp2_sift[m.trainIdx].pt for m in sift_match.good_matches])
dst_s = np.float32([kp1_sift[m.queryIdx].pt for m in sift_match.good_matches])
H_s, inliers_s, info_s = find_homography_ransac(src_s, dst_s, threshold=5.0)

# --- ORB RANSAC ---
src_o = np.float32([kp2_orb[m.trainIdx].pt for m in good_orb])
dst_o = np.float32([kp1_orb[m.queryIdx].pt for m in good_orb])

if len(good_orb) >= 4:
    H_o, inliers_o, info_o = find_homography_ransac(src_o, dst_o, threshold=5.0)
else:
    H_o, inliers_o, info_o = None, None, {'n_iters': 0, 'n_inliers': 0, 'inlier_ratio': 0, 'mean_error': float('inf')}

print(f"{'':>8} {'Matches':>10} {'Inliers':>10} {'Ratio':>10} {'Iters':>8} {'Error':>10}")
print(f"{'SIFT':>8} {len(src_s):>10} {info_s['n_inliers']:>10} {info_s['inlier_ratio']:>9.1%} {info_s['n_iters']:>8} {info_s['mean_error']:>9.2f}px")
if H_o is not None:
    print(f"{'ORB':>8} {len(src_o):>10} {info_o['n_inliers']:>10} {info_o['inlier_ratio']:>9.1%} {info_o['n_iters']:>8} {info_o['mean_error']:>9.2f}px")
else:
    print(f"{'ORB':>8} {len(src_o):>10} {'FAILED':>10}")

## 7. Summary Table

In [ ]:
from IPython.display import display, Markdown

sift_row = [
    'SIFT',
    f'{len(kp1_sift)} / {len(kp2_sift)}',
    f'{len(sift_match.good_matches)}',
    f'{info_s["inlier_ratio"]:.1%}',
    f'{info_s["mean_error"]:.2f}px',
    f'{info_s["n_iters"]}',
    f'{t_sift:.3f}s',
]

orb_row = [
    'ORB',
    f'{len(kp1_orb)} / {len(kp2_orb)}',
    f'{len(good_orb)}',
    f'{info_o["inlier_ratio"]:.1%}' if H_o else 'N/A',
    f'{info_o["mean_error"]:.2f}px' if H_o else 'N/A',
    f'{info_o["n_iters"]}' if H_o else 'N/A',
    f'{t_orb:.3f}s',
]

md = """
| Detector | Keypoints | Matches | Inlier Rate | Mean Error | RANSAC Iters | Time |
|----------|-----------|---------|-------------|------------|--------------|------|
"""
md += f"| {' | '.join(sift_row)} |\n"
md += f"| {' | '.join(orb_row)} |\n"

display(Markdown(md))

print(f"\n{'='*60}")
print("  Conclusion")
print(f"{'='*60}")
print(f"  SIFT: Higher quality matches ({info_s['inlier_ratio']:.1%} inlier), lower error")
print(f"        Best for: high-quality stitching, scale-varying scenes")
print(f"  ORB:  {t_sift/t_orb:.1f}× faster, fewer but usable matches")
print(f"        Best for: real-time applications, fixed-scale scenes")